<a href="https://colab.research.google.com/github/aligreo/TriEncoder-Unet-Project/blob/main/msseg_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install SimpleITK monai nibabel matplotlib numpy

In [ ]:
import os
import random
import warnings
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    NormalizeIntensityd,
    ConcatItemsd,
    DeleteItemsd,
    EnsureTyped,
    CropForegroundd,
    Lambdad,
)
from monai.data import PersistentDataset, DataLoader

from msseg_utils import (
    unzip_if_needed,
    collect_dataset,
    stratified_split,
    binarize_label
)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Preprocessing Functions

In [ ]:
def load_msseg_data(zip_path, extract_dir):
    """Extracts and collects the MSSEG dataset."""
    root = unzip_if_needed(zip_path, extract_dir)
    files = collect_dataset(root, "MSSEG")
    print(f"MSSEG cases found: {len(files)}")
    return files

def create_msseg_transforms():
    """Defines the MONAI preprocessing pipeline."""
    return Compose([
        LoadImaged(keys=["flair", "t1", "t2", "label"]),
        EnsureChannelFirstd(keys=["flair", "t1", "t2", "label"]),
        Orientationd(keys=["flair", "t1", "t2", "label"], axcodes="RAS"),
        Spacingd(
            keys=["flair", "t1", "t2", "label"],
            pixdim=(1.0, 1.0, 1.0),
            mode=("bilinear", "bilinear", "bilinear", "nearest"),
            padding_mode="zeros",
        ),
        Lambdad(keys="label", func=binarize_label),
        CropForegroundd(keys=["flair", "t1", "t2", "label"], source_key="flair"),
        NormalizeIntensityd(keys=["flair", "t1", "t2"], nonzero=True, channel_wise=True),
        ConcatItemsd(keys=["flair", "t1", "t2"], name="image", dim=0),
        DeleteItemsd(keys=["flair", "t1", "t2"]),
        EnsureTyped(keys=["image", "label"]),
    ])

def setup_data_loader(data_files, transforms, cache_dir, val_fraction=0.2, seed=42, batch_size=1):
    """Splits data and sets up the MONAI DataLoader."""
    random.seed(seed)
    np.random.seed(seed)
    
    train_files, val_files = stratified_split(data_files, val_fraction=val_fraction, seed=seed)
    
    dataset = PersistentDataset(
        data=val_files, 
        transform=transforms, 
        cache_dir=cache_dir
    )
    
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    print(f"Data loader ready. Train: {len(train_files)}, Val: {len(val_files)}")
    return loader

## Execution

In [ ]:
# Configuration
ZIP_PATH = "/content/drive/MyDrive/MSSEG-Training.zip"
EXTRACT_DIR = "/content/MSSEG-Training"
CACHE_DIR = "/content/cache_val_msseg"

# Steps
data_files = load_msseg_data(ZIP_PATH, EXTRACT_DIR)
transforms = create_msseg_transforms()
val_loader = setup_data_loader(data_files, transforms, CACHE_DIR)